# 404 — Program Annotation

## Objective

Biologically characterize the three frozen cross-system consensus transcriptomic
representations using prespecified pathway- and process-level annotation
frameworks.

The goal is to determine which biological processes are preferentially
represented across the ranked gene-weight profiles of each consensus program
and to establish an interpretable annotation layer for downstream functional,
pharmacogenomic, and perturbational analyses.

## Analytical status

Notebook 404 operates exclusively on the frozen consensus representations
constructed in notebook 401.

Consensus-program identities, orientations, gene weights, source-system
correspondences, lineage-robustness results, and the negative/limited
epigenetic-regulator enrichment results from notebook 403 are upstream evidence
and are not modified here.

Annotation results cannot be used to redefine, rescue, exclude, reweight, or
rename a consensus program post hoc.

## Annotation framework

Biological annotation will be performed on the complete ranked consensus-gene
profiles rather than on arbitrarily selected top-gene cutoffs.

Gene-set resources and statistical parameters will be frozen before inspecting
enrichment results. Multiple-testing correction will be applied within
prespecified annotation families.

Interpretation will prioritize:

- effect magnitude and rank consistency;
- coherent biological themes rather than isolated significant terms;
- concordance or divergence across the three consensus programs;
- source-system context where necessary; and
- explicit separation between biological annotation and mechanistic inference.

## Scope and methodological boundary

This notebook does not:

- modify consensus-program construction;
- search additional annotation databases after observing results;
- interpret pathway enrichment as causal pathway activation;
- infer regulator activity or signaling direction without supporting evidence;
- use pharmacogenomic, dependency, or perturbational outcomes;
- promote individual genes or pathways as validated targets; or
- reinterpret annotation significance as biological validation.

The resulting annotations are descriptive and hypothesis-generating. They are
intended to clarify the biological identity of the frozen consensus programs
and provide a stable interpretation layer for subsequent phases.

## Expected outputs

Notebook 404 will publish a compact set of downstream-consumable biological
annotation artifacts under `data/processed/consensus_programs/`.

Intermediate enrichment matrices, permutation null distributions, redundant
gene-set catalogs, and exploratory figures will not be persisted unless they
become necessary downstream.

In [1]:
# =============================================================================
# Imports
# =============================================================================

import numpy as np
import pandas as pd

from statsmodels.stats.multitest import multipletests

from pancancer_epigenetics.utils.paths import Paths

In [2]:
# =============================================================================
# Input and output directories
# =============================================================================

CONSENSUS_PROGRAM_DIR = Paths.consensus_programs
OUTPUT_DIR = Paths.consensus_programs
MSIGDB_DIR = Paths.msigdb

In [3]:
# =============================================================================
# Authoritative program-annotation input paths
# =============================================================================

CONSENSUS_CATALOG_PATH = (
    CONSENSUS_PROGRAM_DIR
    / "401_consensus_transcriptomic_program_catalog.csv"
)

CONSENSUS_GENE_WEIGHTS_PATH = (
    CONSENSUS_PROGRAM_DIR
    / "401_consensus_transcriptomic_gene_weights.csv"
)

In [4]:
# =============================================================================
# Load authoritative consensus-program inputs
# =============================================================================

consensus_catalog = pd.read_csv(
    CONSENSUS_CATALOG_PATH
)

consensus_gene_weights = pd.read_csv(
    CONSENSUS_GENE_WEIGHTS_PATH
)

In [5]:
# =============================================================================
# Freeze consensus-gene annotation universe
# =============================================================================

consensus_gene_universe = (
    consensus_gene_weights[["gene_symbol"]]
    .drop_duplicates()
    .sort_values("gene_symbol")
    .reset_index(drop=True)
)

consensus_gene_universe.shape

(2389, 1)

In [6]:
# =============================================================================
# Freeze signed consensus-program ranking
# =============================================================================

ranked_consensus_gene_weights = (
    consensus_gene_weights[
        [
            "consensus_program_id",
            "gene_symbol",
            "consensus_weight",
        ]
    ]
    .sort_values(
        ["consensus_program_id", "consensus_weight"],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

In [7]:
# =============================================================================
# Freeze biological-annotation design
# =============================================================================

MSIGDB_RELEASE = "v2026.1.Hs"

ANNOTATION_COLLECTIONS = {
    "HALLMARK": {
        "filename": "h.all.v2026.1.Hs.symbols.gmt",
        "analysis_tier": "primary",
    },
    "REACTOME": {
        "filename": "c2.cp.reactome.v2026.1.Hs.symbols.gmt",
        "analysis_tier": "secondary",
    },
    "GO_BP": {
        "filename": "c5.go.bp.v2026.1.Hs.symbols.gmt",
        "analysis_tier": "exploratory",
    },
}

MIN_GENE_SET_SIZE = 10
MAX_GENE_SET_SIZE = 500
N_PERMUTATIONS = 10_000
RANDOM_SEED = 404
FDR_ALPHA = 0.05